# Regime-Based Bitcoin Accumulation Strategy

This notebook implements the regime-based strategy.
The strategy uses StackSats MVRV, StackSats Momentum, and a custom SMA 90-day strategy as candidate strategies. DCA is used as the benchmark.

In [1]:
# ============================================================
# Cell 1: Imports and configuration
# ============================================================
# This cell imports required libraries and defines all strategy settings.
# It also sets the train/test periods, window size, lookback periods,
# fallback strategy, and candidate strategies.

import polars as pl
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path

from stacksats.runner.core import StrategyRunner
from stacksats.strategy_types import ExportConfig
from stacksats.strategies.stable.mvrv.core import MVRVStrategy
from stacksats.strategies.stable.signals.momentum import MomentumStrategy


# Path to prepared StackSats BTC analytics dataset.
btc_path = Path.home() / ".stacksats" / "data" / "bitcoin_analytics.parquet"

# Budget used per 365-day window.
TOTAL_BUDGET_USD = 1000.0

# Train and test periods.
TRAIN_START = "2018-01-01"
TRAIN_END = "2023-12-31"
TEST_START = "2024-01-01"
TEST_END = "2025-12-31"

# Each evaluation window is 365 days.
WINDOW_SIZE = 365

# Lookbacks used for momentum, SMA, drawdown, and regime classification.
MOMENTUM_LOOKBACK = 60
SMA_LOOKBACK = 90
DRAWDOWN_LOOKBACK = 180
REGIME_LOOKBACK = 180

# Unique lookback values used to create rolling features.
LOOKBACK_DAYS = sorted({
    MOMENTUM_LOOKBACK,
    SMA_LOOKBACK,
    DRAWDOWN_LOOKBACK,
    REGIME_LOOKBACK,
})

# Small floor to avoid zero or negative allocation signals.
SIGNAL_FLOOR = 1e-8

# Minimum number of days required for a regime to be evaluated.
MIN_REGIME_DAYS = 20

# Tolerance used to decide whether a result is better, worse, or tied.
STATUS_TOLERANCE_PCT = 1e-6

# Fallback strategy used if a regime appears in test but was not seen in training.
FALLBACK_STRATEGY = "sma_90d_weight"

# Candidate strategies used in regime selection.
# DCA is benchmark only and is not selected as a candidate here.
CANDIDATE_COLS = [
    "stacksats_mvrv_weight",
    "stacksats_momentum_weight",
    f"sma_{SMA_LOOKBACK}d_weight",
]

In [2]:
# ============================================================
# Cell 2: Helper functions
# ============================================================
# This cell defines general helper functions used across the notebook.
# These functions are reused for labeling results, formatting chart text,
# trimming complete windows, normalizing weights, and summarizing results.

def get_status_from_pct_diff(pct_diff, tolerance=STATUS_TOLERANCE_PCT):
    """
    Convert percentage improvement into a simple status label.

    Parameters
    ----------
    pct_diff : float
        Percentage difference of strategy performance versus DCA.
    tolerance : float
        Small threshold used to avoid classifying tiny numerical differences
        as meaningful wins or losses.

    Returns
    -------
    str
        'better' if strategy beats DCA, 'worse' if it underperforms,
        and 'tie' if the difference is within tolerance.
    """

    # If improvement is greater than tolerance, strategy is better.
    if pct_diff > tolerance:
        return "better"

    # If improvement is below negative tolerance, strategy is worse.
    if pct_diff < -tolerance:
        return "worse"

    # Otherwise treat it as no meaningful difference.
    return "tie"


def format_arrow_text(extra_spd, improvement_pct):
    """
    Create chart annotation text for positive or negative SPD improvement.

    Parameters
    ----------
    extra_spd : float
        Extra sats per dollar compared with DCA.
    improvement_pct : float
        Percentage improvement compared with DCA.

    Returns
    -------
    tuple[str, str]
        Formatted text and color name. Positive values use a green upward
        arrow, while negative values use a red downward arrow.
    """

    # Positive result: green upward arrow.
    if extra_spd >= 0:
        return f"▲ +{extra_spd:,.2f} sats/$ ({improvement_pct:+.2f}%)", "green"

    # Negative result: red downward arrow.
    return f"▼ {extra_spd:,.2f} sats/$ ({improvement_pct:+.2f}%)", "red"


def trim_full_windows(df: pd.DataFrame, window_size: int = WINDOW_SIZE):
    """
    Keep only complete fixed-length windows.

    Parameters
    ----------
    df : pd.DataFrame
        Input dataframe sorted by date.
    window_size : int
        Number of rows per evaluation window. In this strategy it is 365.

    Returns
    -------
    tuple[pd.DataFrame, int, int]
        Trimmed dataframe, number of complete windows, and number of dropped
        remainder rows.
    """

    # Count how many complete windows exist.
    n_full = len(df) // window_size

    # Number of rows that fit into complete windows.
    n_eval = n_full * window_size

    # Number of leftover rows that will be dropped.
    remainder = len(df) - n_eval

    # Return trimmed dataframe, number of windows, and dropped rows.
    return df.iloc[:n_eval].copy(), n_full, remainder


def build_simple_normalized_weights(signal_multiplier, signal_floor=SIGNAL_FLOOR):
    """
    Convert signal multipliers into normalized daily allocation weights.

    Parameters
    ----------
    signal_multiplier : array-like
        Raw signal strength or multiplier values.
    signal_floor : float
        Minimum allowed signal value to prevent zero or negative weights.

    Returns
    -------
    np.ndarray
        Daily allocation weights that sum to 1.
    """

    # Convert input into a numpy array.
    signal_multiplier = np.asarray(signal_multiplier, dtype=float)

    # Prevent empty signal arrays.
    if len(signal_multiplier) == 0:
        raise ValueError("Empty signal array.")

    # Replace NaN and infinity values with 0.
    clean_signal = np.nan_to_num(
        signal_multiplier,
        nan=0.0,
        posinf=0.0,
        neginf=0.0,
    )

    # Apply a small floor so no value is exactly zero or negative.
    clean_signal = np.maximum(clean_signal, signal_floor)

    # If all signals are invalid or zero, fall back to uniform weights.
    if clean_signal.sum() <= 0:
        return np.full(len(clean_signal), 1.0 / len(clean_signal))

    # Normalize so all daily weights sum to 1.
    return clean_signal / clean_signal.sum()


def summarize_spd_like_composite(window_summary_df):
    """
    Summarize performance across all 365-day windows.

    Parameters
    ----------
    window_summary_df : pd.DataFrame
        Window-level summary dataframe containing strategy_sats, dca_sats,
        strategy_spd, dca_spd, and result columns.

    Returns
    -------
    dict
        Summary metrics including total sats, SPD sums, improvement percentage,
        wins, losses, ties, and win rate.
    """

    # Number of full 365-day windows.
    n_windows = len(window_summary_df)

    # Sum strategy and DCA sats-per-dollar across windows.
    strategy_spd_sum = window_summary_df["strategy_spd"].sum()
    dca_spd_sum = window_summary_df["dca_spd"].sum()

    # Extra sats-per-dollar gained or lost vs DCA.
    extra_spd_sum = strategy_spd_sum - dca_spd_sum

    # Ratio of strategy SPD to DCA SPD.
    spd_ratio = strategy_spd_sum / dca_spd_sum

    # Percentage improvement over DCA.
    improvement_pct = (spd_ratio - 1.0) * 100.0

    # Sum total sats accumulated by strategy and DCA.
    strategy_sats = window_summary_df["strategy_sats"].sum()
    dca_sats = window_summary_df["dca_sats"].sum()

    # Extra sats accumulated vs DCA.
    extra_sats = strategy_sats - dca_sats

    # Count windows where strategy beat, lost to, or tied DCA.
    wins = int((window_summary_df["result"] == "better").sum())
    losses = int((window_summary_df["result"] == "worse").sum())
    ties = int((window_summary_df["result"] == "tie").sum())

    # Window win rate.
    win_rate_pct = wins / n_windows * 100.0 if n_windows > 0 else 0.0

    # Return one summary dictionary.
    return {
        "n_windows": n_windows,
        "wins": wins,
        "losses": losses,
        "ties": ties,
        "win_rate_pct": win_rate_pct,

        "strategy_sats": strategy_sats,
        "dca_sats": dca_sats,
        "extra_sats_vs_dca": extra_sats,

        "strategy_spd_sum": strategy_spd_sum,
        "dca_spd_sum": dca_spd_sum,
        "extra_spd_sum_vs_dca": extra_spd_sum,

        "strategy_spd_avg": strategy_spd_sum / n_windows,
        "dca_spd_avg": dca_spd_sum / n_windows,
        "extra_spd_avg_vs_dca": extra_spd_sum / n_windows,

        "spd_ratio": spd_ratio,
        "improvement_pct": improvement_pct,
    }


def build_log_tick_values_and_text():
    """
    Build custom y-axis tick values and labels for the BTC log-price chart.

    Returns
    -------
    tuple[list[int], list[str]]
        Tick values and corresponding display labels.
    """

    # Actual BTC price values used as tick positions.
    tickvals = [
        3000, 4000, 5000, 6000, 7000, 8000, 9000,
        10000,
        20000, 30000, 40000, 50000, 60000, 70000, 80000, 90000,
        100000,
    ]

    # Labels displayed on the chart.
    ticktext = [
        "3", "4", "5", "6", "7", "8", "9",
        "10k",
        "2", "3", "4", "5", "6", "7", "8", "9",
        "100k",
    ]

    return tickvals, ticktext

In [3]:
# ============================================================
# Cell 3: Load BTC data
# ============================================================
# This cell loads the prepared Bitcoin analytics parquet file.
# It checks that all required columns exist before moving forward.

if not btc_path.exists():
    raise FileNotFoundError(f"Could not find: {btc_path}")

btc_df = (
    pl.read_parquet(btc_path)
    .with_columns(pl.col("date").cast(pl.Datetime))
    .sort("date")
)

required_cols = [
    "date",
    "price_usd",
    "mvrv",
    "adjusted_sopr",
    "adjusted_sopr_7d_ema",
    "realized_cap_growth_rate",
    "market_cap_growth_rate",
]

missing_cols = [col for col in required_cols if col not in btc_df.columns]

if missing_cols:
    raise ValueError(f"Missing required columns from bitcoin_analytics.parquet: {missing_cols}")

print("Loaded BTC rows:", btc_df.height)

display(
    btc_df.select(
        pl.col("date").min().alias("min_date"),
        pl.col("date").max().alias("max_date"),
    )
)

Loaded BTC rows: 5689


min_date,max_date
datetime[μs],datetime[μs]
2010-08-16 00:00:00,2026-03-13 00:00:00


In [4]:
# ============================================================
# Cell 4: StackSats strategy export setup
# ============================================================
# This cell sets up the StackSats strategy runner.
# It exports daily weights from built-in StackSats MVRV and Momentum strategies.

runner = StrategyRunner()

stacksats_strategy_objects = {
    "stacksats_mvrv_weight": MVRVStrategy(),
    "stacksats_momentum_weight": MomentumStrategy(),
}

# Cache prevents recomputing weights for the same strategy/window repeatedly.
_export_cache = {}


def export_stacksats_weights_for_window(
    strategy_key: str,
    window_df: pd.DataFrame,
    full_btc_df: pl.DataFrame,
):
    """
    Export StackSats strategy weights for one 365-day window.

    Parameters
    ----------
    strategy_key : str
        Key identifying which StackSats strategy to run.
    window_df : pd.DataFrame
        Current 365-day window dataframe.
    full_btc_df : pl.DataFrame
        Full BTC analytics dataframe in Polars format.

    Returns
    -------
    np.ndarray
        Normalized daily weights for the selected StackSats strategy.
    """

    # Get current window start and end dates.
    window_start = pd.to_datetime(window_df["date"].min()).strftime("%Y-%m-%d")
    window_end = pd.to_datetime(window_df["date"].max()).strftime("%Y-%m-%d")

    # Cache key is based on strategy and date range.
    cache_key = (strategy_key, window_start, window_end)

    # If already computed, return cached version.
    if cache_key in _export_cache:
        return _export_cache[cache_key].copy()

    # Create StackSats export config for this window.
    config = ExportConfig(
        range_start=window_start,
        range_end=window_end,
    )

    # Filter BTC data to the current window only.
    window_btc_df = (
        full_btc_df
        .filter(
            (pl.col("date") >= pd.to_datetime(window_start)) &
            (pl.col("date") <= pd.to_datetime(window_end)) &
            pl.col("price_usd").is_not_null()
        )
        .sort("date")
    )

    # Stop if no data exists for this window.
    if window_btc_df.is_empty():
        raise ValueError(f"No BTC data available for {window_start} to {window_end}")

    # Select the requested StackSats strategy object.
    strategy = stacksats_strategy_objects[strategy_key]

    # Run StackSats export.
    export_obj = runner.export(
        strategy,
        config,
        btc_df=window_btc_df,
    )

    # Convert export result to dataframe.
    weights = export_obj.to_dataframe()

    # Ensure output is a Polars dataframe.
    if not isinstance(weights, pl.DataFrame):
        weights = pl.from_pandas(weights)

    # Cast date columns into datetime format.
    weights = weights.with_columns([
        pl.col("start_date").cast(pl.Datetime),
        pl.col("end_date").cast(pl.Datetime),
        pl.col("date").cast(pl.Datetime),
    ])

    # Get the most recent export window from StackSats output.
    latest_end = weights.select(pl.col("end_date").max()).item()

    # Select only date and weight for that latest window.
    one_window = (
        weights
        .filter(pl.col("end_date") == latest_end)
        .sort("date")
        .select(["date", "weight"])
        .rename({"weight": "raw_weight"})
        .to_pandas()
    )

    one_window["date"] = pd.to_datetime(one_window["date"])

    # Merge exported weights back to the original window dates.
    merged = (
        window_df[["date"]]
        .merge(one_window, on="date", how="left")
        .sort_values("date")
        .reset_index(drop=True)
    )

    # If any dates are missing weights, stop and show examples.
    if merged["raw_weight"].isna().any():
        missing_dates = (
            merged.loc[merged["raw_weight"].isna(), "date"]
            .dt.strftime("%Y-%m-%d")
            .head(10)
            .tolist()
        )

        raise ValueError(
            f"Missing StackSats weights for {strategy_key} "
            f"from {window_start} to {window_end}. "
            f"Example missing dates: {missing_dates}"
        )

    # Normalize StackSats raw weights so the window sums to 1.
    final_weights = build_simple_normalized_weights(
        merged["raw_weight"].values
    )

    # Store in cache.
    _export_cache[cache_key] = final_weights.copy()

    return final_weights

In [5]:
# ============================================================
# Cell 5: Feature engineering
# ============================================================
# This cell creates rolling features used for regime classification.
# It creates SMA values, returns, rolling highs, SMA ratios, and drawdown features.

feature_exprs = []

for d in LOOKBACK_DAYS:
    feature_exprs.extend([
        pl.col("price_usd")
        .rolling_mean(window_size=d, min_samples=max(3, int(d * 0.30)))
        .alias(f"price_{d}d_sma"),

        pl.col("price_usd")
        .pct_change(d)
        .alias(f"btc_return_{d}d"),

        pl.col("price_usd")
        .rolling_max(window_size=d, min_samples=max(3, int(d * 0.30)))
        .alias(f"rolling_high_{d}d"),
    ])

btc_df = btc_df.with_columns(feature_exprs)

ratio_exprs = []

for d in LOOKBACK_DAYS:
    ratio_exprs.extend([
        (pl.col("price_usd") / pl.col(f"price_{d}d_sma"))
        .alias(f"price_{d}d_sma_ratio"),

        ((pl.col("price_usd") / pl.col(f"rolling_high_{d}d")) - 1)
        .alias(f"drawdown_{d}d"),
    ])

btc_df = btc_df.with_columns(ratio_exprs)

btc_data = btc_df.to_pandas()
btc_data["date"] = pd.to_datetime(btc_data["date"])

feature_cols = [
    "price_usd",
    "mvrv",
    "adjusted_sopr",
    "adjusted_sopr_7d_ema",
    "realized_cap_growth_rate",
    "market_cap_growth_rate",
]

for d in LOOKBACK_DAYS:
    feature_cols.extend([
        f"price_{d}d_sma",
        f"price_{d}d_sma_ratio",
        f"btc_return_{d}d",
        f"drawdown_{d}d",
    ])

btc_data = (
    btc_data
    .dropna(subset=feature_cols)
    .sort_values("date")
    .reset_index(drop=True)
)

print("Cleaned date range:")
print(btc_data["date"].min(), "to", btc_data["date"].max())
print("Cleaned rows:", len(btc_data))

Cleaned date range:
2011-08-16 00:00:00 to 2026-03-13 00:00:00
Cleaned rows: 5324


In [6]:
# ============================================================
# Cell 6: Regime classification
# ============================================================
# This cell labels every day with a combined market regime.
# The regime combines BTC trend, SOPR, MVRV valuation, and capital-growth behavior.

def classify_btc_onchain_regime(row):
    """
    Classify each day into a combined BTC/on-chain regime.

    Parameters
    ----------
    row : pd.Series
        One row of the engineered BTC dataframe.

    Returns
    -------
    str
        Combined regime label in the format:
        BTC trend regime | SOPR regime | MVRV regime | capital-growth regime
    """

    # Price vs moving average features.
    sma_90_ratio = row[f"price_{SMA_LOOKBACK}d_sma_ratio"]
    sma_180_ratio = row[f"price_{REGIME_LOOKBACK}d_sma_ratio"]

    # Momentum and drawdown features.
    return_momentum = row[f"btc_return_{MOMENTUM_LOOKBACK}d"]
    return_90d = row[f"btc_return_{SMA_LOOKBACK}d"]
    drawdown_180d = row[f"drawdown_{DRAWDOWN_LOOKBACK}d"]

    # On-chain valuation and stress features.
    mvrv = row["mvrv"]
    sopr_ema = row["adjusted_sopr_7d_ema"]
    realized_growth = row["realized_cap_growth_rate"]
    market_growth = row["market_cap_growth_rate"]

    # Classify BTC trend / drawdown state.
    if drawdown_180d <= -0.50:
        btc_regime = "BTC Severe Drawdown"
    elif drawdown_180d <= -0.30:
        btc_regime = "BTC Deep Drawdown"
    elif sma_180_ratio >= 1.05 and return_90d > 0:
        btc_regime = "BTC Bull"
    elif sma_90_ratio <= 0.95 and return_90d < 0:
        btc_regime = "BTC Bear"
    elif sma_90_ratio < 1.0 and return_momentum > 0:
        btc_regime = "BTC Recovery"
    else:
        btc_regime = "BTC Neutral"

    # Classify SOPR market behavior.
    # SOPR below 1 generally indicates coins being spent at a loss.
    if sopr_ema < 0.98:
        sopr_regime = "SOPR Stress"
    elif sopr_ema > 1.02:
        sopr_regime = "SOPR Profit"
    else:
        sopr_regime = "SOPR Neutral"

    # Classify valuation using MVRV.
    if mvrv < 1.0:
        valuation_regime = "Low MVRV"
    elif mvrv > 2.5:
        valuation_regime = "High MVRV"
    else:
        valuation_regime = "Normal MVRV"

    # Compare realized-cap growth against market-cap growth.
    if realized_growth > market_growth:
        cap_regime = "Realized Growth Leading"
    else:
        cap_regime = "Market Growth Leading"

    # Combine all regime dimensions into one label.
    return btc_regime + " | " + sopr_regime + " | " + valuation_regime + " | " + cap_regime


btc_data["combined_regime"] = btc_data.apply(classify_btc_onchain_regime, axis=1)

display(
    btc_data[["date", "price_usd", "combined_regime"]].head()
)

,date,price_usd,combined_regime
0,2011-08-16,11.05,BTC Severe Drawdown | SOPR Neutral | Normal MV...
1,2011-08-17,10.88,BTC Severe Drawdown | SOPR Neutral | Normal MV...
2,2011-08-18,10.90,BTC Severe Drawdown | SOPR Neutral | Normal MV...
3,2011-08-19,11.40,BTC Severe Drawdown | SOPR Neutral | Normal MV...
4,2011-08-20,11.49,BTC Severe Drawdown | SOPR Neutral | Normal MV...


In [7]:
# ============================================================
# Cell 7: Train/test split
# ============================================================
# This cell splits data into train and test periods.
# It also trims each split into complete 365-day evaluation windows.

raw_train_df = btc_data[
    (btc_data["date"] >= pd.to_datetime(TRAIN_START)) &
    (btc_data["date"] <= pd.to_datetime(TRAIN_END))
].copy().reset_index(drop=True)

raw_test_df = btc_data[
    (btc_data["date"] >= pd.to_datetime(TEST_START)) &
    (btc_data["date"] <= pd.to_datetime(TEST_END))
].copy().reset_index(drop=True)

train_eval_df, n_train_windows, train_remainder = trim_full_windows(raw_train_df, WINDOW_SIZE)
test_eval_df, n_test_windows, test_remainder = trim_full_windows(raw_test_df, WINDOW_SIZE)

split_summary_df = pd.DataFrame([
    {
        "split": "train",
        "start_date": raw_train_df["date"].min(),
        "end_date": raw_train_df["date"].max(),
        "rows_total": len(raw_train_df),
        "rows_eval": len(train_eval_df),
        "windows": n_train_windows,
        "remainder_dropped": train_remainder,
        "budget_rule": "$1,000 per 365-day training window",
    },
    {
        "split": "test",
        "start_date": raw_test_df["date"].min(),
        "end_date": raw_test_df["date"].max(),
        "rows_total": len(raw_test_df),
        "rows_eval": len(test_eval_df),
        "windows": n_test_windows,
        "remainder_dropped": test_remainder,
        "budget_rule": "$1,000 per 365-day test window",
    },
])

display(split_summary_df)

,split,start_date,end_date,rows_total,rows_eval,windows,remainder_dropped,budget_rule
0,train,2018-01-01,2023-12-31,2191,2190,6,1,"$1,000 per 365-day training window"
1,test,2024-01-01,2025-12-31,731,730,2,1,"$1,000 per 365-day test window"


In [8]:
# ============================================================
# Cell 8: Candidate strategy weights
# ============================================================
# This cell creates daily weights for each candidate strategy.
# Candidate strategies are StackSats MVRV, StackSats Momentum, and custom SMA 90D.

def create_candidate_strategy_weights_simple(data):
    """
    Create daily allocation weights for all candidate strategies.

    Parameters
    ----------
    data : pd.DataFrame
        One 365-day window of BTC data with regime and engineered features.

    Returns
    -------
    pd.DataFrame
        Original data plus strategy weight columns:
        - dca_weight
        - stacksats_mvrv_weight
        - stacksats_momentum_weight
        - sma_90d_weight
    """

    # Sort data by date and reset index.
    df = data.copy().sort_values("date").reset_index(drop=True)

    # Number of days in the current window.
    n = len(df)

    # Stop if the input window is empty.
    if n == 0:
        raise ValueError("No data available.")

    # Uniform DCA benchmark weight.
    # This is not used as a candidate strategy in this version.
    df["dca_weight"] = 1.0 / n

    # Candidate 1: StackSats MVRV strategy.
    # This comes directly from the StackSats built-in MVRV strategy.
    df["stacksats_mvrv_weight"] = export_stacksats_weights_for_window(
        strategy_key="stacksats_mvrv_weight",
        window_df=df,
        full_btc_df=btc_df,
    )

    # Candidate 2: StackSats Momentum strategy.
    # This comes directly from the StackSats built-in Momentum strategy.
    df["stacksats_momentum_weight"] = export_stacksats_weights_for_window(
        strategy_key="stacksats_momentum_weight",
        window_df=df,
        full_btc_df=btc_df,
    )

    # Candidate 3: Custom SMA 90-day strategy.
    # price_90d_sma_ratio < 1 means BTC price is below the 90-day SMA.
    # In that case, sma_signal becomes positive and allocation increases.
    sma_signal = (1.0 - df[f"price_{SMA_LOOKBACK}d_sma_ratio"]).clip(-1, 1)

    # Convert SMA signal into a multiplier.
    # 1.50 controls how strongly the SMA signal changes allocation.
    sma_multiplier = np.maximum(SIGNAL_FLOOR, 1.0 + 1.50 * sma_signal)

    # Normalize SMA multipliers into daily weights that sum to 1.
    df[f"sma_{SMA_LOOKBACK}d_weight"] = build_simple_normalized_weights(
        sma_multiplier.values
    )

    return df

In [9]:
# ============================================================
# Cell 9: Evaluate strategies by regime
# ============================================================
# This cell evaluates each candidate strategy within each regime across training windows.
# It compares each candidate against DCA using sats accumulated and sats per dollar.

def evaluate_strategies_by_regime_in_365_windows(
    data,
    total_budget_usd=TOTAL_BUDGET_USD,
):
    """
    Evaluate all candidate strategies inside each regime for every training window.

    Parameters
    ----------
    data : pd.DataFrame
        Training dataframe trimmed to complete 365-day windows.
    total_budget_usd : float
        Budget allocated per 365-day window.

    Returns
    -------
    pd.DataFrame
        Regime-level strategy performance versus DCA.
    """

    # Number of full 365-day windows.
    n_windows = len(data) // WINDOW_SIZE

    # Store evaluation rows here.
    rows = []

    # Loop through each 365-day window.
    for window_idx in range(n_windows):
        start = window_idx * WINDOW_SIZE
        end = start + WINDOW_SIZE

        # Slice one window.
        window_data = data.iloc[start:end].copy().reset_index(drop=True)

        # Create candidate strategy weights for the window.
        df = create_candidate_strategy_weights_simple(data=window_data)

        # Evaluate each regime separately.
        for regime, regime_df in df.groupby("combined_regime"):

            # Skip regimes with too few days.
            if len(regime_df) < MIN_REGIME_DAYS:
                continue

            # Calculate DCA sats for this regime subset.
            dca_sats = (
                regime_df["dca_weight"]
                * total_budget_usd
                / regime_df["price_usd"]
                * 100_000_000
            ).sum()

            # Compare every candidate strategy against DCA.
            for col in CANDIDATE_COLS:

                # Calculate strategy sats for this regime subset.
                strategy_sats = (
                    regime_df[col]
                    * total_budget_usd
                    / regime_df["price_usd"]
                    * 100_000_000
                ).sum()

                # Convert total sats into sats per dollar.
                strategy_spd = strategy_sats / total_budget_usd
                dca_spd = dca_sats / total_budget_usd

                # Extra sats and extra SPD vs DCA.
                extra_sats = strategy_sats - dca_sats
                extra_spd = strategy_spd - dca_spd

                # Relative performance vs DCA.
                spd_ratio = strategy_spd / dca_spd
                improvement_pct = (spd_ratio - 1.0) * 100.0

                # Save one evaluation row.
                rows.append({
                    "train_window": window_idx + 1,
                    "combined_regime": regime,
                    "days": len(regime_df),
                    "strategy": col,

                    "strategy_sats": strategy_sats,
                    "dca_sats": dca_sats,
                    "extra_sats_vs_dca": extra_sats,

                    "strategy_spd": strategy_spd,
                    "dca_spd": dca_spd,
                    "extra_spd_vs_dca": extra_spd,
                    "spd_ratio": spd_ratio,
                    "improvement_pct": improvement_pct,

                    "status": get_status_from_pct_diff(improvement_pct),
                })

    return pd.DataFrame(rows)


train_regime_results_df = evaluate_strategies_by_regime_in_365_windows(
    data=train_eval_df,
    total_budget_usd=TOTAL_BUDGET_USD,
)

display(
    train_regime_results_df
    .sort_values(["combined_regime", "improvement_pct"], ascending=[True, False])
    .round(6)
)

,train_window,combined_regime,days,strategy,strategy_sats,dca_sats,extra_sats_vs_dca,strategy_spd,dca_spd,extra_spd_vs_dca,spd_ratio,improvement_pct,status
81,6,BTC Bear | SOPR Neutral | Normal MVRV | Market...,34,stacksats_mvrv_weight,6.920313e+05,3.565695e+05,3.354618e+05,692.031285,356.569504,335.461782,1.940803,94.080334,better
83,6,BTC Bear | SOPR Neutral | Normal MVRV | Market...,34,sma_90d_weight,4.763443e+05,3.565695e+05,1.197748e+05,476.344335,356.569504,119.774831,1.335909,33.590879,better
82,6,BTC Bear | SOPR Neutral | Normal MVRV | Market...,34,stacksats_momentum_weight,4.020210e+05,3.565695e+05,4.545146e+04,402.020967,356.569504,45.451464,1.127469,12.746873,better
16,2,BTC Bull | SOPR Neutral | Normal MVRV | Market...,34,stacksats_momentum_weight,1.113544e+06,9.909298e+05,1.226144e+05,1113.544231,990.929811,122.614419,1.123737,12.373673,better
86,6,BTC Bull | SOPR Neutral | Normal MVRV | Market...,87,sma_90d_weight,9.100113e+05,8.173162e+05,9.269504e+04,910.011251,817.316210,92.695041,1.113414,11.341393,better
...,...,...,...,...,...,...,...,...,...,...,...,...,...
33,2,BTC Severe Drawdown | SOPR Stress | Low MVRV |...,48,stacksats_mvrv_weight,3.649935e+06,3.649935e+06,-0.000000e+00,3649.935336,3649.935336,-0.000000,1.000000,-0.000000,tie
9,1,BTC Severe Drawdown | SOPR Stress | Low MVRV |...,34,stacksats_mvrv_weight,9.300616e+03,2.548114e+06,-2.538813e+06,9.300616,2548.113949,-2538.813333,0.003650,-99.635000,worse
12,1,BTC Severe Drawdown | SOPR Stress | Normal MVR...,75,stacksats_mvrv_weight,9.302768e+06,2.908188e+06,6.394580e+06,9302.768064,2908.187843,6394.580221,3.198820,219.881953,better
14,1,BTC Severe Drawdown | SOPR Stress | Normal MVR...,75,sma_90d_weight,3.386078e+06,2.908188e+06,4.778903e+05,3386.078177,2908.187843,477.890334,1.164326,16.432581,better


In [10]:
# ============================================================
# Cell 10: Candidate win rate by regime
# ============================================================
# This cell summarizes how often each candidate strategy beats DCA across regime cases.

candidate_regime_winrate_df = (
    train_regime_results_df
    .groupby("strategy", as_index=False)
    .agg(
        regime_cases=("status", "count"),
        wins=("status", lambda x: (x == "better").sum()),
        losses=("status", lambda x: (x == "worse").sum()),
        ties=("status", lambda x: (x == "tie").sum()),
        avg_improvement_pct=("improvement_pct", "mean"),
    )
)

candidate_regime_winrate_df["win_rate_pct"] = (
    candidate_regime_winrate_df["wins"]
    / candidate_regime_winrate_df["regime_cases"]
    * 100.0
)

display(
    candidate_regime_winrate_df
    .sort_values("win_rate_pct", ascending=False)
    .round(6)
)

,strategy,regime_cases,wins,losses,ties,avg_improvement_pct,win_rate_pct
1,stacksats_momentum_weight,32,17,15,0,0.836960,53.125
0,sma_90d_weight,32,15,17,0,2.726122,46.875
2,stacksats_mvrv_weight,32,10,20,2,11.922741,31.250
